In [2]:
import os


SR = 22050
N_FFT = 2048
HOP = 256

def wav_path_to_float(wav_path: str) -> float:
    # e.g. dataset/dynamic/dynamic_mix_max_0p1.wav -> "dynamic_mix_max_0p1.wav"
    filename = os.path.basename(wav_path)
    # -> "0p1"
    num_str = filename.replace('.wav', '').split('_')[-1]
    # replace 'p' with '.' and convert
    return float(num_str.replace('p', '.'))

def seconds_to_frames(seconds, sample_rate=SR, hop_length=HOP):
    return int(round((seconds * sample_rate) / hop_length))

In [4]:
import numpy as np
import librosa
from sklearn.decomposition import PCA
from pathlib import Path

dynamic_dir = Path("./dataset/dynamic")
wav_files = sorted(dynamic_dir.glob("*.wav"))

all_data = []
frameValues = []

for wav_path in wav_files:
    audio, sr = librosa.load(wav_path, sr=SR)
    D = librosa.stft(audio, n_fft=N_FFT, hop_length=HOP)
    specgram_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

    # PCA over frequency bins (transpose keeps frequencies as features)
    pca_specgram = PCA(n_components=0.9).fit(specgram_db.T)
    transformed = pca_specgram.transform(specgram_db.T)

    # Mean-pool across time frames to get a fixed-length vector per file
    all_data.append(transformed.mean(axis=0))
    frameValues.append(seconds_to_frames(wav_path_to_float(wav_path)))

y = np.vstack(frameValues)
X = np.vstack(all_data)
print("X shape:", X.shape)


X shape: (50, 2)


In [5]:
import pandas as pd

print("y:", y)
print("X:", X)

pd.DataFrame(X).head(50)


y: [[  9]
 [ 17]
 [ 26]
 [ 34]
 [ 43]
 [ 52]
 [ 60]
 [ 69]
 [ 78]
 [ 86]
 [ 95]
 [103]
 [112]
 [121]
 [129]
 [138]
 [146]
 [155]
 [164]
 [172]
 [181]
 [189]
 [198]
 [207]
 [215]
 [224]
 [233]
 [241]
 [250]
 [258]
 [267]
 [276]
 [284]
 [293]
 [301]
 [310]
 [319]
 [327]
 [336]
 [345]
 [353]
 [362]
 [370]
 [379]
 [388]
 [396]
 [405]
 [413]
 [422]
 [431]]
X: [[ 3.11411568e-04 -5.66004237e-05]
 [ 4.36977716e-04  1.88272432e-04]
 [-8.63845868e-04 -4.56772366e-04]
 [-3.64793726e-04  3.25391913e-04]
 [ 7.97236280e-04 -1.60506635e-04]
 [ 7.88732956e-04  6.31816627e-04]
 [ 7.60766387e-04  1.61327443e-05]
 [ 1.50603781e-04  5.27514785e-04]
 [-6.51356590e-04  1.84035525e-04]
 [ 1.81404801e-04 -2.43160379e-04]
 [-9.12314979e-04  9.61882397e-05]
 [ 1.08965707e-03 -4.72290230e-05]
 [ 2.53872247e-04  1.15866998e-04]
 [ 5.10201016e-06  2.43231243e-05]
 [ 6.46254630e-05  5.84605289e-07]
 [ 1.88160237e-04 -6.79795558e-05]
 [ 2.77398183e-04  1.26401705e-04]
 [-3.70746071e-04  1.94106673e-04]
 [-5.69063064

,0,1
0,0.000311,-5.660042e-05
1,0.000437,1.882724e-04
2,-0.000864,-4.567724e-04
3,-0.000365,3.253919e-04
4,0.000797,-1.605066e-04
5,0.000789,6.318166e-04
6,0.000761,1.613274e-05
7,0.000151,5.275148e-04
8,-0.000651,1.840355e-04
9,0.000181,-2.431604e-04


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(5)


In [8]:
model.fit(X_train, y_train)
print("Model trained successfully")

Model trained successfully


/Users/micahlim/Desktop/NMF_separation/.venv/lib/python3.13/site-packages/sklearn/neighbors/_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Users/micahlim/Desktop/NMF_separation/.venv/lib/python3.13/site-packages/sklearn/neighbors/_base.py:501: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


In [9]:
y_pred = model.predict(X_test)

In [10]:
from sklearn.metrics import mean_squared_error
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))

from sklearn.metrics import mean_absolute_error
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))

from sklearn.metrics import r2_score
print("R^2 Score:", r2_score(y_test, y_pred))

Mean Squared Error: 47913.0
Mean Absolute Error: 189.4
R^2 Score: -4.170343016912901
